In [1]:
import os
import itertools
import numpy as np
import matplotlib.pyplot as plt
import quimb.tensor as qtn

In [2]:
def random_clifford_t_block(qc, q1, q2, add_t, p_t=0.2):
    for q in [q1, q2]:
        r = np.random.randint(4)
        if r == 0:
            qc.h(q)
        elif r == 1:
            qc.s(q)
        elif r == 2:
            qc.h(q)
            qc.s(q)

    if np.random.rand() < 0.5:
        qc.cx(q1, q2)
    else:
        qc.cx(q2, q1)

    if add_t:
        if np.random.rand() < p_t:
            qc.t(q1)
        if np.random.rand() < p_t:
            qc.t(q2)


def build_circuit_2d(Lx, Ly, d, add_t, p_t=0.2, max_bond=64, cutoff=1e-10):
    N = Lx * Ly
    qc = qtn.CircuitMPS(N, max_bond=max_bond, cutoff=cutoff)

    def idx(x, y):
        return y * Lx + x

    for layer in range(d):
        direction = layer % 2
        offset = (layer // 2) % 2

        if direction == 0:
            for y in range(Ly):
                for x in range(offset, Lx - 1, 2):
                    random_clifford_t_block(qc, idx(x, y), idx(x + 1, y), add_t=add_t, p_t=p_t)
        else:
            for x in range(Lx):
                for y in range(offset, Ly - 1, 2):
                    random_clifford_t_block(qc, idx(x, y), idx(x, y + 1), add_t=add_t, p_t=p_t)

    return qc


def projected_cp_mps_small(qc, n_measured):
    """
    Projected ensemble collision probability using a SMALL, fixed measured
    subsystem (n_measured qubits, indices 0..n_measured-1).

    Formula (derived via LU-invariance on the LARGE projected subsystem A):

        (2^n_projected)^2 * sum_x  |<x_B, 0_A | psi>|^4 / p(x_B)

    where:
    - sum is over ALL 2^n_measured outcomes x of the measured subsystem (small!)
    - |<x_B, 0_A|psi>|^2 = p(x_B, y_A=0): joint probability of measured=x AND
      projected=all-zero, extracted via a SINGLE amplitude query per x
    - p(x_B) comes from a single compute_marginal call returning the full 2^n_measured
      marginal distribution at once
    - 2 factors of 2^n_projected: one from the LU-twirl collapsing sum_y→single y=0,
      one from the standard collision-probability normalization convention

    Why this is better than the N/2-split equation-3 approach:
    - LOWER VARIANCE: averages over all 2^n_measured x-outcomes within each circuit
      instance (vs relying on a single noisy x=0 proxy); variance still exists due
      to the LU-approximation on the y-side, but is reduced by the x-averaging
    - SMALLER MPS BOND REQUIREMENT: the cut at position n_measured (small) in the
      MPS chain has bounded entanglement entropy (capped by n_measured); however,
      note that internal bonds deeper in the chain (within the large projected
      subsystem) are NOT bounded by this, so max_bond still controls accuracy for
      the GLOBAL state representation
    - No additional approximation beyond the MPS state truncation itself; each
      amplitude() and compute_marginal() call is exact given the current MPS

    Valid in expectation over an LU-invariant ensemble at sufficient circuit depth.
    Aggregate with np.nanmean so NaN samples don't wipe out the whole average.
    """
    N = qc.N
    n_projected = N - n_measured

    marginal = qc.compute_marginal(where=list(range(n_measured)))

    total = 0.0
    for x_bits in itertools.product([0, 1], repeat=n_measured):
        p_x = marginal[x_bits]
        if p_x <= 1e-15:
            continue
        bitstring = ''.join(map(str, x_bits)) + '0' * n_projected
        amp = qc.amplitude(bitstring)
        total += abs(amp) ** 4 / p_x

    if not np.isfinite(total) or total <= 0:
        return np.nan

    return (2 ** n_projected) ** 2 * total

In [3]:
# Validation: gate-log replay, comparing ensemble averages of this formula
# against the exact equation-(2) statevector reference at N=6, d=10.
# Expected: means should match within ~1-2 standard errors.

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def _build_gate_log(N, d, add_t, p_t):
    log = []

    def block(q1, q2):
        for q in [q1, q2]:
            r = np.random.randint(4)
            if r == 0: log.append(('h', q))
            elif r == 1: log.append(('s', q))
            elif r == 2: log.append(('h', q)); log.append(('s', q))
        if np.random.rand() < 0.5: log.append(('cx', q1, q2))
        else: log.append(('cx', q2, q1))
        if add_t:
            if np.random.rand() < p_t: log.append(('t', q1))
            if np.random.rand() < p_t: log.append(('t', q2))

    for layer in range(d):
        start = layer % 2
        for i in range(start, N - 1, 2):
            block(i, i + 1)
    return log


N_val, d_val, n_measured_val = 6, 10, 2
n_projected_val = N_val - n_measured_val
samples_val = 200

my_vals, ref_vals = [], []
for s in range(samples_val):
    np.random.seed(s)
    log = _build_gate_log(N_val, d_val, True, 0.2)

    circ = qtn.CircuitMPS(N_val)
    for g in log:
        if g[0] == 'h': circ.h(g[1])
        elif g[0] == 's': circ.s(g[1])
        elif g[0] == 't': circ.t(g[1])
        elif g[0] == 'cx': circ.cx(g[1], g[2])

    marginal = circ.compute_marginal(where=list(range(n_measured_val)))
    total = 0.0
    for x_bits in itertools.product([0, 1], repeat=n_measured_val):
        p_x = marginal[x_bits]
        if p_x <= 1e-15: continue
        bitstring = ''.join(map(str, x_bits)) + '0' * n_projected_val
        amp = circ.amplitude(bitstring)
        total += abs(amp) ** 4 / p_x
    my_vals.append((2 ** n_projected_val) ** 2 * total)

    qc_ref = QuantumCircuit(N_val)
    for g in log:
        if g[0] == 'h': qc_ref.h(g[1])
        elif g[0] == 's': qc_ref.s(g[1])
        elif g[0] == 't': qc_ref.t(g[1])
        elif g[0] == 'cx': qc_ref.cx(g[1], g[2])
    probs = Statevector.from_instruction(qc_ref).probabilities()
    probs_2d = probs.reshape(2 ** n_projected_val, 2 ** n_measured_val)
    p_x_ref = probs_2d.sum(axis=0)
    mask = p_x_ref > 1e-15
    ref_vals.append((2 ** n_projected_val) * np.sum((probs_2d ** 2).sum(axis=0)[mask] / p_x_ref[mask]))

print(f"Validation: N={N_val}, d={d_val}, n_measured={n_measured_val}, samples={samples_val}")
print(f"  This formula mean = {np.nanmean(my_vals):.4f}  std = {np.nanstd(my_vals):.4f}")
print(f"  Exact eq-2  mean = {np.mean(ref_vals):.4f}  std = {np.std(ref_vals):.4f}")
print(f"  (should match within ~1-2 SEM = {np.nanstd(my_vals)/np.sqrt(samples_val):.3f})")

Validation: N=6, d=10, n_measured=2, samples=200
  This formula mean = 2.4863  std = 4.7516
  Exact eq-2  mean = 2.0506  std = 0.7897
  (should match within ~1-2 SEM = 0.336)


In [4]:
# Vary N, fixed depth
L_values = list(range(2, 9))   # N = 4 .. 64
d = 10
add_t = True
n_measured = 2        # kept SMALL and FIXED — 2^n_measured = 4 amplitude calls per sample
samples_per_n = 50    # more samples needed vs N/2-split (higher per-instance variance)
max_bond = 64
output_dir = "results/mps_small"
os.makedirs(output_dir, exist_ok=True)

avg_prob, std_prob, total_qubits = [], [], []

for L in L_values:
    N = L * L
    n_projected = N - n_measured
    vals = []

    for _ in range(samples_per_n):
        qc = build_circuit_2d(L, L, d, add_t, p_t=0.15, max_bond=max_bond)
        vals.append(projected_cp_mps_small(qc, n_measured))

    n_dropped = int(np.sum(np.isnan(vals)))
    print(f"L={L}, N={N}, n_measured={n_measured}, n_projected={n_projected}, "
          f"dropped {n_dropped}/{samples_per_n}")

    avg_prob.append(np.nanmean(vals))
    std_prob.append(np.nanstd(vals))
    total_qubits.append(N)

    fig, ax = plt.subplots()
    ax.errorbar(total_qubits, avg_prob, yerr=std_prob, marker='o', capsize=4,
                label=f"d={d}, n_meas={n_measured}, chi<={max_bond}")
    ax.set_xlabel("N (total qubits)")
    ax.set_ylabel(r"$(2^{n_B})^2 \sum_x |\langle x,0|\psi\rangle|^4 / p(x)$")
    ax.set_yscale('log')
    ax.set_title(f"2D Brickwork: Projected Ensemble CP (small-measured MPS, d={d})")
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.savefig(os.path.join(output_dir, f"mps_small_d{d}_upto_N{N}.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  saved mps_small_d{d}_upto_N{N}.png")

plt.errorbar(total_qubits, avg_prob, yerr=std_prob, marker='o', capsize=4,
             label=f"d={d}, n_meas={n_measured}, chi<={max_bond}")
plt.xlabel("N (total qubits)")
plt.ylabel(r"$(2^{n_B})^2 \sum_x |\langle x,0|\psi\rangle|^4 / p(x)$")
plt.yscale('log')
plt.title(f"2D Brickwork: Projected Ensemble CP (small-measured MPS, d={d})")
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()

L=2, N=4, n_measured=2, n_projected=2, dropped 0/50
  saved mps_small_d10_upto_N4.png
L=3, N=9, n_measured=2, n_projected=7, dropped 0/50
  saved mps_small_d10_upto_N9.png
L=4, N=16, n_measured=2, n_projected=14, dropped 0/50
  saved mps_small_d10_upto_N16.png


c:\Users\Daniel\Desktop\research\.venv\Lib\site-packages\quimb\tensor\decomp.py:1028: UserWarning: Got: Internal algorithm failed to converge., falling back to scipy gesvd driver.
  warnings.warn(f"Got: {e}, falling back to scipy gesvd driver.")


L=5, N=25, n_measured=2, n_projected=23, dropped 0/50
  saved mps_small_d10_upto_N25.png
L=6, N=36, n_measured=2, n_projected=34, dropped 0/50
  saved mps_small_d10_upto_N36.png
L=7, N=49, n_measured=2, n_projected=47, dropped 1/50
  saved mps_small_d10_upto_N49.png


ZeroDivisionError: float division by zero

In [ ]:
# Vary depth, fixed N
L = 5          # N = 25
d_values = list(range(1, 20))
add_t = True
n_measured = 2
samples_per_d = 50
max_bond = 64
output_dir = "results/mps_small"
os.makedirs(output_dir, exist_ok=True)

N = L * L
n_projected = N - n_measured

avg_prob, std_prob, completed_depths = [], [], []

for d in d_values:
    vals = []

    for _ in range(samples_per_d):
        qc = build_circuit_2d(L, L, d, add_t, p_t=0.15, max_bond=max_bond)
        vals.append(projected_cp_mps_small(qc, n_measured))

    n_dropped = int(np.sum(np.isnan(vals)))
    print(f"d={d}, N={N}, n_measured={n_measured}, n_projected={n_projected}, "
          f"dropped {n_dropped}/{samples_per_d}")

    avg_prob.append(np.nanmean(vals))
    std_prob.append(np.nanstd(vals))
    completed_depths.append(d)

    fig, ax = plt.subplots()
    ax.errorbar(completed_depths, avg_prob, yerr=std_prob, marker='o', capsize=4,
                label=f"N={N}, n_meas={n_measured}, chi<={max_bond}")
    ax.set_xlabel("depth")
    ax.set_ylabel(r"$(2^{n_B})^2 \sum_x |\langle x,0|\psi\rangle|^4 / p(x)$")
    ax.set_yscale('log')
    ax.set_title(f"2D Brickwork: Projected Ensemble CP (small-measured MPS, N={N})")
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.savefig(os.path.join(output_dir, f"mps_small_N{N}_upto_d{d}.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  saved mps_small_N{N}_upto_d{d}.png")

plt.errorbar(completed_depths, avg_prob, yerr=std_prob, marker='o', capsize=4,
             label=f"N={N}, n_meas={n_measured}, chi<={max_bond}")
plt.xlabel("depth")
plt.ylabel(r"$(2^{n_B})^2 \sum_x |\langle x,0|\psi\rangle|^4 / p(x)$")
plt.yscale('log')
plt.title(f"2D Brickwork: Projected Ensemble CP (small-measured MPS, N={N})")
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()